# 💬 Módulo 06 — Conversaciones y Sesiones

> **Objetivo:** Mantener el contexto de conversación entre múltiples turnos usando `AgentSession`.

## ¿Por qué sesiones?

Sin sesión, cada llamada a `agent.run()` es **independiente** — el agente no recuerda nada de turnos anteriores.

```
Llamada 1: "Mi color favorito es azul"  →  "Genial!"
Llamada 2: "¿Cuál es mi color favorito?"  →  "No tengo esa información"  ❌
```

Con sesión, el historial se mantiene:

```
Turno 1 (sesión X): "Mi color favorito es azul"  →  "Genial!"
Turno 2 (sesión X): "¿Cuál es mi color favorito?"  →  "Es azul"  ✅
```

## API

```python
session = agent.create_session()           # crear sesión
await agent.run("...", session=session)    # usarla en cada turno
session.state["custom_key"] = "value"      # estado personalizado tipo dict
```


## ⚙️ Setup inicial

Esta celda carga la configuración de Azure OpenAI desde `appsettings.Development.json`
(o variables de entorno) y crea el cliente que reutilizaremos durante todo el notebook.

> 💡 Solo necesitas ejecutar esta celda **una vez** al abrir el notebook.


In [6]:
# Aseguramos que la raíz del proyecto esté en sys.path para importar `helpers`
import sys
from pathlib import Path

_ROOT = Path.cwd()
# Si abriste el notebook desde la carpeta notebooks/, sube un nivel
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from agent_framework import Agent
from helpers.config import create_chat_client
print("✅ Helpers cargados. Endpoint:", create_chat_client().__class__.__name__)


✅ Helpers cargados. Endpoint: OpenAIChatClient


## 1️⃣ Sin sesión (o sesiones distintas) → sin memoria

Demostramos que con sesiones separadas el agente no comparte el historial.


In [2]:
agent = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un asistente útil. Responde las preguntas directamente. "
        "Si no sabes algo del contexto, di 'No tengo esa información'."
    ),
)

session_a = agent.create_session()
session_b = agent.create_session()

r1 = await agent.run("Mi color favorito es el azul.", session=session_a)
print(f"Sesión A → {r1.text}\n")

r2 = await agent.run("¿Cuál es mi color favorito?", session=session_b)
print(f"Sesión B → {r2.text}")
print("\n✅ Con sesiones separadas, cada llamada es independiente.")


Sesión A → ¡Qué bien! El azul es un color que evoca calma, serenidad y confianza.

Sesión B → No tengo esa información.

✅ Con sesiones separadas, cada llamada es independiente.


## 2️⃣ Una sesión → contexto persistente entre turnos

La misma sesión `agent.create_session()` reutilizada en cada llamada conserva el historial completo.


In [3]:
agent = Agent(
    client=create_chat_client(),
    instructions="Eres un tutor de matemáticas. Recuerda todas las interacciones previas.",
)

session = agent.create_session()

r1 = await agent.run("Mi nombre es Carlos y estoy aprendiendo álgebra.", session=session)
print(f"Turno 1 → {r1.text}\n")

r2 = await agent.run("¿Cuál es mi nombre?", session=session)
print(f"Turno 2 → {r2.text}\n")

r3 = await agent.run("¿Qué materia estoy aprendiendo?", session=session)
print(f"Turno 3 → {r3.text}")
print("\n✅ La sesión mantuvo el contexto entre los 3 turnos.")


Turno 1 → ¡Hola, Carlos! Es un gusto ayudarte con el álgebra. ¿Qué tema específico estás viendo ahora? ¿Ecuaciones, expresiones algebraicas, factorización, sistemas de ecuaciones, o algo más? ¡Estoy aquí para resolver tus dudas! 😊

Turno 2 → Tu nombre es **Carlos**. 😊 ¿En qué te ayudo con álgebra hoy?

Turno 3 → Estás aprendiendo **álgebra**. 😊 Si tienes alguna duda o ejercicio que quieras resolver, ¡aquí estoy para ayudarte!

✅ La sesión mantuvo el contexto entre los 3 turnos.


## 3️⃣ Estado personalizado en `session.state`

`session.state` es un `dict[str, Any]` donde puedes guardar información de tu aplicación
(user_id, preferencias, contadores, etc.).

Es el equivalente Python del `StateBag` de C#.


In [7]:
agent = Agent(client=create_chat_client(), instructions="Eres un asistente útil.")
session = agent.create_session()

# Guardar estado personalizado
session.state["user_id"] = "USR-12345"
session.state["session_type"] = "workshop_demo"
session.state["interaction_count"] = 0

await agent.run("¡Hola!", session=session)
session.state["interaction_count"] += 1

print("✅ Estado personalizado:")
for k, v in session.state.items():
    print(f"   {k}: {v}")


✅ Estado personalizado:
   user_id: USR-12345
   session_type: workshop_demo
   interaction_count: 1


## 4️⃣ Almacenamiento con `InMemoryHistoryProvider`

Por defecto, `agent.create_session()` mantiene el historial internamente.
Pero si queremos **control explícito** sobre cómo se almacenan los mensajes, usamos un `InMemoryHistoryProvider`.

📚 **Doc oficial:** [Storage](https://learn.microsoft.com/en-us/agent-framework/agents/conversations/storage?pivots=programming-language-python)

### ¿Por qué usar un HistoryProvider?

| Sin provider explícito | Con `InMemoryHistoryProvider` |
|---|---|
| El agente maneja el historial internamente | Tú controlas el almacenamiento |
| No puedes aplicar compactación | Puedes encadenar `CompactionProvider` |
| No puedes serializar/restaurar fácilmente | Puedes usar `session.to_dict()` / `from_dict()` |

```python
from agent_framework import InMemoryHistoryProvider

history = InMemoryHistoryProvider("memory", load_messages=True)
agent = Agent(client=client, instructions="...", context_providers=[history])
```

> 💡 `load_messages=True` significa que este provider **carga** el historial en el contexto del modelo.
> Solo un provider debe tener `load_messages=True` a la vez.

In [8]:
from agent_framework import InMemoryHistoryProvider

# Crear un history provider explícito
history = InMemoryHistoryProvider("memory", load_messages=True)

agent = Agent(
    client=create_chat_client(),
    instructions="Eres un asistente de cocina. Recuerda los ingredientes que mencione el usuario.",
    context_providers=[history],
)

session = agent.create_session()

r1 = await agent.run("Tengo tomates, cebolla y ajo.", session=session)
print(f"Turno 1 → {r1.text}\n")

r2 = await agent.run("¿Qué ingredientes te mencioné?", session=session)
print(f"Turno 2 → {r2.text}")
print("\n✅ InMemoryHistoryProvider mantiene el historial entre turnos.")

Turno 1 → Perfecto, tienes tomates, cebolla y ajo. ¿Quieres que te sugiera una receta con esos ingredientes, o tienes algo en mente? 

Turno 2 → Me mencionaste **tomates**, **cebolla** y **ajo**. 😊

✅ InMemoryHistoryProvider mantiene el historial entre turnos.


## 5️⃣ Serializar y restaurar sesiones

Puedes **persistir** una sesión completa con `session.to_dict()` y restaurarla después con `AgentSession.from_dict()`.
Esto es útil para guardar conversaciones en disco, base de datos o caché.

```
Sesión activa  →  session.to_dict()  →  JSON / DB / Redis
                                          ↓
Nueva instancia  ←  AgentSession.from_dict(data)  ←  JSON / DB / Redis
```

> ⚠️ **Importante:** Debes restaurar la sesión con la **misma configuración** de agente/providers que la creó.

In [9]:
from agent_framework import AgentSession
import json

# Serializar la sesión actual a un dict
serialized = session.to_dict()
print("📦 Sesión serializada:")
print(json.dumps(serialized, indent=2, default=str, ensure_ascii=False)[:500])
print("...\n")

# Restaurar la sesión desde el dict
restored_session = AgentSession.from_dict(serialized)

# Continuar la conversación con la sesión restaurada
r3 = await agent.run("Sugiere una receta con esos ingredientes.", session=restored_session)
print(f"Turno 3 (sesión restaurada) → {r3.text}")
print("\n✅ La sesión fue serializada, restaurada, y el agente recuerda el contexto.")

📦 Sesión serializada:
{
  "type": "session",
  "session_id": "ff414385-d7ad-47f2-a374-f582dc137a5d",
  "service_session_id": "resp_08315dc57d47f383006a1de28237288197bd4410d20bfdeb94",
  "state": {
    "memory": {
      "messages": [
        {
          "type": "message",
          "role": "user",
          "contents": [
            {
              "type": "text",
              "text": "Tengo tomates, cebolla y ajo.",
              "additional_properties": {}
            }
          ],
          "additional_properties
...

Turno 3 (sesión restaurada) → ¡Claro! Con **tomates**, **cebolla** y **ajo**, puedes preparar una deliciosa **salsa de tomate casera**. Aquí tienes la receta:

### **Salsa de tomate casera**
Ideal para pasta, arroz, acompañar carnes o como base de otras preparaciones.

#### **Ingredientes que necesitas:**
- **Tomates**: Lo que tengas, puede ser entre 3 y 4 tomates medianos.
- **Cebolla**: 1 cebolla (mediana o grande, según tu preferencia).
- **Ajo**: 2 o 3 dientes (aj

## 6️⃣ Compactación — Controlar el tamaño del historial

📚 **Doc oficial:** [Compaction](https://learn.microsoft.com/en-us/agent-framework/agents/conversations/compaction?pivots=programming-language-python)

### ¿Por qué compactar?

A medida que la conversación crece, el historial puede:
- **Exceder la ventana de contexto** del modelo → error
- **Aumentar costos** → más tokens = más caro
- **Aumentar latencia** → más tokens de entrada = más lento

La compactación **reduce** el historial preservando el contexto importante.

### Estrategias disponibles

| Estrategia | Agresividad | Preserva contexto | Usa LLM |
|---|---|---|---|
| `SlidingWindowStrategy` | 🔴 Alta | Baja — descarta grupos antiguos | No |
| `TruncationStrategy` | 🔴 Alta | Baja — descarta por tokens | No |
| `ToolResultCompactionStrategy` | 🟢 Baja | Alta — colapsa resultados de tools | No |
| `SummarizationStrategy` | 🟡 Media | Media — resume con LLM | **Sí** |
| `TokenBudgetComposedStrategy` | ⚙️ Configurable | Pipeline de varias estrategias | Depende |

### Arquitectura

```
                    ┌─────────────────────────────────────────┐
                    │           CompactionProvider            │
                    │                                         │
  agent.run() ──→   │  before_strategy → [modelo] → after_strategy  │  ──→ respuesta
                    │        ↑                      ↑         │
                    │   compacta antes          compacta       │
                    │   de enviar al LLM     lo que se guarda  │
                    └─────────────────────────────────────────┘
```

> ⚠️ **Experimental en Python:** Las estrategias se importan desde `agent_framework._compaction`.

### 6a. `SlidingWindowStrategy` — Ventana deslizante

La estrategia más simple: **conserva solo los últimos N grupos** de mensajes y descarta los anteriores.
Los mensajes del sistema (`instructions`) siempre se preservan.

- Cada mensaje (user o assistant) es **1 grupo**
- `keep_last_groups=2` → solo conserva el último turno (1 user + 1 assistant)
- Los datos mencionados en turnos anteriores **desaparecen** del contexto

```python
from agent_framework._compaction import SlidingWindowStrategy
sliding = SlidingWindowStrategy(keep_last_groups=2)  # solo el último turno
```

> 🔍 **Observa:** Después de 4 turnos de datos, el agente ya no puede responder preguntas sobre los primeros turnos.

In [12]:
from agent_framework import CompactionProvider, InMemoryHistoryProvider
from agent_framework._compaction import SlidingWindowStrategy

# Configurar: conservar solo los últimos 2 grupos (= 1 turno user+assistant)
# Esto significa que el agente SOLO verá el turno más reciente
history = InMemoryHistoryProvider("memory", load_messages=True)
compaction = CompactionProvider(
    before_strategy=SlidingWindowStrategy(keep_last_groups=2),
    history_source_id=history.source_id,
)

agent = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un asistente con memoria limitada. Responde brevemente. "
        "Si no tienes información previa del usuario, di exactamente 'No tengo esa información'."
    ),
    context_providers=[history, compaction],
)

session = agent.create_session()

# Primero damos información en varios turnos
datos = [
    "Me llamo Ana.",
    "Vivo en Madrid.",
    "Trabajo como diseñadora.",
    "Mi comida favorita es la paella.",
]

for i, msg in enumerate(datos, 1):
    r = await agent.run(msg, session=session)
    print(f"Turno {i}: {msg}")
    print(f"  → {r.text}\n")

# Ahora preguntamos por datos antiguos — la ventana deslizante ya los descartó
preguntas = [
    "¿Cómo me llamo?",
    "¿Dónde vivo?",
    "¿Cuál es mi comida favorita?",
]

print("=" * 60)
print("Ahora preguntamos por datos que ya salieron de la ventana:\n")
for q in preguntas:
    r = await agent.run(q, session=session)
    print(f"Pregunta: {q}")
    print(f"  → {r.text}\n")

print("✅ SlidingWindowStrategy(keep_last_groups=2) solo ve el último turno.")
print("   Los datos anteriores (nombre, ciudad) se descartaron del contexto.")

Turno 1: Me llamo Ana.
  → ¡Hola, Ana! ¿En qué puedo ayudarte?

Turno 2: Vivo en Madrid.
  → ¡Genial, Ana! ¿En qué puedo ayudarte hoy?

Turno 3: Trabajo como diseñadora.
  → ¡Qué interesante, Ana! ¿Cómo puedo ayudarte hoy?

Turno 4: Mi comida favorita es la paella.
  → ¡La paella es deliciosa, Ana! ¿En qué puedo ayudarte hoy?

Ahora preguntamos por datos que ya salieron de la ventana:

Pregunta: ¿Cómo me llamo?
  → Te llamas Ana.

Pregunta: ¿Dónde vivo?
  → No tengo esa información.

Pregunta: ¿Cuál es mi comida favorita?
  → No tengo esa información.

✅ SlidingWindowStrategy(keep_last_groups=2) solo ve el último turno.
   Los datos anteriores (nombre, ciudad) se descartaron del contexto.


### 6b. `SummarizationStrategy` — Resumir con LLM

A diferencia de `SlidingWindowStrategy` que **pierde información**, esta estrategia **resume** los turnos
antiguos con un LLM. El agente **sí recuerda** los hechos clave, pero usando **muchos menos tokens**.

| SlidingWindow | Summarization |
|---|---|
| ❌ Pierde datos antiguos | ✅ Preserva hechos clave en un resumen |
| Rápido, sin costo extra | Usa un LLM adicional (costo extra) |
| Ideal para contexto desechable | Ideal para conversaciones largas importantes |

### ¿Cómo funciona internamente?

```
Turno 1-3 (6 mensajes)  →  Se activa SummarizationStrategy
                            ↓
                     Los 2 mensajes más viejos se RESUMEN en 1 mensaje
                     Los 4 más recientes se CONSERVAN intactos
                            ↓
                     Resultado: 5 mensajes (1 resumen + 4 recientes)
                     en vez de 6 originales
```

### Parámetros clave

- **`target_count`** — Cuántos mensajes recientes **conservar intactos** (no se resumen)
- **`threshold`** — Cuántos mensajes **extras** debe haber para activar el resumen

**Fórmula:** El resumen se activa cuando hay ≥ `target_count + threshold` mensajes.
Se resumen **todos los mensajes más viejos que `target_count`**.

| `target_count` | `threshold` | Se activa con | Se resumen | Se conservan intactos |
|---|---|---|---|---|
| 4 | 2 | 6+ mensajes | Todo excepto los 4 más recientes | Los 4 más recientes |
| 2 | 2 | 4+ mensajes | Todo excepto los 2 más recientes | Los 2 más recientes |
| 6 | 4 | 10+ mensajes | Todo excepto los 6 más recientes | Los 6 más recientes |

> 🎛️ **Para resumir más agresivamente:** baja `target_count` (conserva menos).
> **Para activar el resumen antes:** baja `threshold`.

### `before_strategy` vs `after_strategy`

El `CompactionProvider` tiene **dos momentos** para compactar:

```
          before_strategy                    after_strategy
               ↓                                  ↓
  [historial] → compacta → [modelo] → respuesta → compacta → [guarda en historial]
```

| Parámetro | Cuándo actúa | Qué hace |
|---|---|---|
| `before_strategy` | **Antes** de enviar al modelo | Reduce tokens enviados al LLM (ahorra costo/latencia) |
| `after_strategy` | **Después** de recibir respuesta | Reduce lo que se **almacena** (ahorra memoria) |

> 💡 Usamos **ambos** para que el efecto sea visible tanto en lo que el modelo recibe como en lo que se guarda.

### `skip_excluded=True`

Cuando la compactación marca mensajes como "excluidos" (`_excluded=True`), el `InMemoryHistoryProvider`
con `skip_excluded=True` **los omite** al cargar — así el modelo no los ve aunque sigan en el dict.

> 💡 **Tip:** Usa un modelo más barato/rápido (como `gpt-4o-mini`) para el summarizer para reducir costos.

In [ ]:
from agent_framework._compaction import SummarizationStrategy

# --- 1. Crear la estrategia de summarización ---
# Usamos el mismo cliente, pero en producción usarías un modelo más barato (gpt-4o-mini)
summarizer_client = create_chat_client()

strategy = SummarizationStrategy(
    client=summarizer_client,
    target_count=4,   # conservar los 4 mensajes más recientes intactos
    threshold=2,      # activar resumen cuando haya 4+2=6 mensajes no-system
)

# --- 2. Configurar history + compaction providers ---
# skip_excluded=True: al cargar historial, omite mensajes marcados como excluidos por la compactación
history = InMemoryHistoryProvider("memory", load_messages=True, skip_excluded=True)

compaction = CompactionProvider(
    before_strategy=strategy,   # compacta lo que se ENVÍA al modelo (ahorra tokens/latencia)
    after_strategy=strategy,    # compacta lo que se GUARDA en el historial (ahorra memoria)
    history_source_id=history.source_id,  # vincula con el history provider correcto
)

# --- 3. Crear agente con ambos providers ---
agent = Agent(
    client=create_chat_client(),
    instructions="Eres un asistente de viajes. Recuerda las preferencias del usuario. Responde en 1 línea.",
    context_providers=[history, compaction],  # orden: history primero, compaction después
)

session = agent.create_session()


# --- 4. Helper para contar mensajes activos (no excluidos) ---
def count_stored_messages() -> int:
    """Cuenta mensajes en el state que NO están marcados como _excluded."""
    provider_state = session.state.get(history.source_id, {})
    msgs = provider_state.get("messages", [])
    return sum(1 for m in msgs if not m.additional_properties.get("_excluded", False))


# --- 5. Simulamos una conversación de 5 turnos ---
mensajes = [
    "Quiero planear un viaje a Japón.",
    "Prefiero viajar en primavera.",
    "Mi presupuesto es de $3000 USD.",
    "Me interesa la cultura y la gastronomía.",
    "No me gustan los tours grupales.",
]

for i, msg in enumerate(mensajes, 1):
    msgs_before = count_stored_messages()

    r = await agent.run(msg, session=session)

    msgs_after = count_stored_messages()

    # Sin compactación esperaríamos +2 por turno (user + assistant).
    # Si msgs_after < expected, la estrategia resumió mensajes antiguos.
    expected = msgs_before + 2
    compacted = f" ← 🗜️ ¡COMPACTADO! (esperado {expected}, real {msgs_after})" if msgs_after < expected and msgs_before > 0 else ""
    print(f"Turno {i}: {msg}")
    print(f"  → {r.text[:150]}{'...' if len(r.text) > 150 else ''}")
    print(f"  📊 Mensajes activos en historial: {msgs_before} → {msgs_after}{compacted}\n")

# --- 6. Pregunta final: ¿el agente recuerda TODO a pesar de la compactación? ---
print("=" * 60)
r = await agent.run("Lista todas mis preferencias de viaje en bullet points.", session=session)
print(f"Pregunta final: Lista todas mis preferencias")
print(f"  → {r.text}\n")
print(f"  📊 Mensajes activos: {count_stored_messages()} (sin compactación serían {len(mensajes) * 2 + 2})")
print("\n✅ A pesar de la compactación, el agente RECUERDA todo (destino, época, presupuesto, intereses).")
print("   Pero el historial tiene MENOS mensajes activos que sin compactación (ahorro de tokens).")

Turno 1: Quiero planear un viaje a Japón.
  → Claro, ¿te interesa explorar las ciudades, la naturaleza o la cultura tradicional en Japón?
  📊 Mensajes activos en historial: 0 → 2

Turno 2: Prefiero viajar en primavera.
  → Perfecto, la primavera es ideal para disfrutar del florecimiento de los cerezos (sakura).
  📊 Mensajes activos en historial: 2 → 4

Turno 3: Mi presupuesto es de $3000 USD.
  → Con $3000 USD, puedes planear un viaje de 7-10 días a Japón, enfocándote en Tokio, Kioto y alrededores.
  📊 Mensajes activos en historial: 4 → 6

Turno 4: Me interesa la cultura y la gastronomía.
  → Perfecto, puedes explorar templos en Kioto, el barrio histórico de Gion y probar sushi, ramen y kaiseki tradicionales.
  📊 Mensajes activos en historial: 6 → 8

Turno 5: No me gustan los tours grupales.
  → Entendido, puedo sugerir recorridos autoguiados y experiencias personalizadas para disfrutar a tu ritmo.
  📊 Mensajes activos en historial: 8 → 10

Pregunta final: Lista todas mis preferencias


### 6c. `TokenBudgetComposedStrategy` — Pipeline de estrategias

La estrategia más poderosa y la **recomendada para producción**: combina múltiples estrategias
en un **pipeline secuencial** con un **presupuesto de tokens** como objetivo.

### ¿Cómo funciona?

1. Se ejecutan las estrategias **en orden** (de gentil a agresiva)
2. Después de cada estrategia, se verifica si el historial ya cabe en el presupuesto de tokens
3. Si ya cabe → **se detiene** (`early_stop=True` por defecto)
4. Si ninguna estrategia logra reducir lo suficiente → **fallback automático** descarta los mensajes más viejos

```
  Historial (50,000 tokens)
       ↓
  1. ToolResultCompactionStrategy  →  colapsa resultados de tools  →  ¿≤16,000? No
       ↓
  2. SummarizationStrategy         →  resume turnos antiguos       →  ¿≤16,000? No
       ↓
  3. SlidingWindowStrategy          →  descarta grupos antiguos     →  ¿≤16,000? Sí ✅ → DETENER
       ↓ (si aún no)
  4. Fallback automático            →  descarta los más viejos
```

### Parámetros clave

- **`token_budget`** — Presupuesto máximo de tokens para el historial
- **`tokenizer`** — Cómo estimar tokens. `CharacterEstimatorTokenizer()` usa ~4 caracteres por token
- **`strategies`** — Lista ordenada de estrategias (de **gentil a agresiva**)
- **`early_stop=True`** (default) — Detenerse cuando el presupuesto se cumple

### ¿Por qué es la mejor?

| Solo SlidingWindow | Solo Summarization | TokenBudgetComposed |
|---|---|---|
| Pierde info siempre | Costo LLM siempre | Solo usa LLM si es necesario |
| No controla tokens | No controla tokens | **Controla presupuesto exacto** |
| Una sola técnica | Una sola técnica | **Combina N técnicas** |

### Orden recomendado de estrategias

| # | Estrategia | Agresividad | Efecto |
|---|---|---|---|
| 1 | `ToolResultCompactionStrategy` | 🟢 Baja | Colapsa resultados de tools en resúmenes cortos |
| 2 | `SummarizationStrategy` | 🟡 Media | Resume turnos antiguos con LLM |
| 3 | `SlidingWindowStrategy` | 🔴 Alta | Descarta grupos antiguos sin piedad |

> 🎯 **Regla de oro:** Siempre pon las estrategias de **menor a mayor agresividad**.
> Así preservas la mayor cantidad de contexto posible.

In [ ]:
from agent_framework._compaction import (
    CharacterEstimatorTokenizer,
    SlidingWindowStrategy,
    SummarizationStrategy,
    TokenBudgetComposedStrategy,
    ToolResultCompactionStrategy,
)

# --- 1. Configurar el tokenizer ---
# CharacterEstimatorTokenizer usa ~4 caracteres = 1 token (heurística rápida)
# En producción puedes usar un tokenizer real como tiktoken
tokenizer = CharacterEstimatorTokenizer()
summarizer_client = create_chat_client()

# --- 2. Crear el pipeline de estrategias (de gentil a agresivo) ---
pipeline = TokenBudgetComposedStrategy(
    token_budget=500,        # presupuesto bajo para que se active rápido en el demo
    tokenizer=tokenizer,
    strategies=[
        ToolResultCompactionStrategy(keep_last_tool_call_groups=1),  # 1. gentil: colapsa tools
        SummarizationStrategy(client=summarizer_client, target_count=2, threshold=2),  # 2. moderado: resume
        SlidingWindowStrategy(keep_last_groups=4),  # 3. agresivo: descarta antiguos
    ],
)

# --- 3. Configurar history + compaction ---
history = InMemoryHistoryProvider("memory", load_messages=True, skip_excluded=True)
compaction = CompactionProvider(
    before_strategy=pipeline,
    after_strategy=pipeline,    # también compactar lo almacenado
    history_source_id=history.source_id,
)

# --- 4. Crear agente ---
agent = Agent(
    client=create_chat_client(),
    instructions="Eres un planificador de eventos. Recuerda los detalles del evento. Responde en 1 línea.",
    context_providers=[history, compaction],
)

session = agent.create_session()


# --- 5. Helper para contar mensajes y estimar tokens ---
def count_and_estimate():
    provider_state = session.state.get(history.source_id, {})
    msgs = provider_state.get("messages", [])
    active = [m for m in msgs if not m.additional_properties.get("_excluded", False)]
    # Estimar tokens con la misma heurística del CharacterEstimatorTokenizer
    total_chars = sum(len(str(getattr(m, "text", ""))) for m in active)
    est_tokens = total_chars // 4
    return len(active), est_tokens


# --- 6. Simulamos una conversación larga ---
mensajes = [
    "Quiero organizar una boda para 150 personas.",
    "El presupuesto es de $50,000 USD.",
    "Será en una playa en Cancún.",
    "La fecha es el 15 de marzo de 2027.",
    "Quiero música en vivo con una banda de jazz.",
    "La comida debe ser menú de 3 tiempos con opciones veganas.",
    "Necesito decoración con flores tropicales.",
    "El dress code es formal tropical.",
    "¿Puedes darme un resumen completo del evento con todos los detalles?",
]

for i, msg in enumerate(mensajes, 1):
    msgs_before, tokens_before = count_and_estimate()

    r = await agent.run(msg, session=session)

    msgs_after, tokens_after = count_and_estimate()

    expected = msgs_before + 2
    compacted = " ← 🗜️ ¡COMPACTADO!" if msgs_after < expected and msgs_before > 0 else ""
    print(f"Turno {i}: {msg}")
    print(f"  → {r.text[:150]}{'...' if len(r.text) > 150 else ''}")
    print(f"  📊 Mensajes: {msgs_before}→{msgs_after} | ~Tokens: {tokens_before}→{tokens_after}{compacted}\n")

# --- 7. Resumen final ---
print("=" * 60)
final_msgs, final_tokens = count_and_estimate()
total_sin_compactacion = len(mensajes) * 2
print(f"📊 Resultado final:")
print(f"   Mensajes activos: {final_msgs} (sin compactación serían ~{total_sin_compactacion})")
print(f"   ~Tokens estimados: {final_tokens} (presupuesto: 500)")
print(f"\n✅ TokenBudgetComposedStrategy mantuvo el historial dentro del presupuesto.")
print(f"   Pipeline: ToolResult → Summarization → SlidingWindow (de gentil a agresivo).")

→ ¡Hola! La compactación es un proceso mediante el cual se reduce el volumen de un material al eliminar espacios vacíos (aire o partículas sueltas) entre sus componentes. Esto se logra aplicando presión, vibración o golpes, lo que aumenta su densidad y estabilidad. Es común en construcción (suelos) y reciclaje (residuos).

✅ TokenBudgetComposedStrategy configurado con pipeline de 3 estrategias.
   El agente aplicará compactación automáticamente cuando el historial crezca.


## 🎯 Resumen

| Capacidad | Cómo se hace |
|-----------|--------------|
| Crear sesión | `session = agent.create_session()` |
| Usarla en un turno | `await agent.run("...", session=session)` |
| Estado personalizado | `session.state["key"] = value` |
| History provider explícito | `InMemoryHistoryProvider("memory", load_messages=True)` |
| Serializar/restaurar sesión | `session.to_dict()` / `AgentSession.from_dict(data)` |
| Compactación por ventana | `SlidingWindowStrategy(keep_last_groups=N)` |
| Compactación por resumen | `SummarizationStrategy(client=..., target_count=N)` |
| Pipeline compuesto | `TokenBudgetComposedStrategy(token_budget=N, strategies=[...])` |
| Registrar compactación | `CompactionProvider(before_strategy=..., history_source_id=...)` |

### 📚 Docs oficiales
- [Storage](https://learn.microsoft.com/en-us/agent-framework/agents/conversations/storage?pivots=programming-language-python)
- [Compaction](https://learn.microsoft.com/en-us/agent-framework/agents/conversations/compaction?pivots=programming-language-python)

**Siguiente módulo →** [Módulo 07 — Context Providers](./07_context_providers.ipynb)